In [2]:
%%writefile config.py
CONFIG = {
    "swing_lookback": 5,
    "structure_lookback": 50,
    "ema_fast": 20,
    "ema_slow": 50,
    "ema_trend": 200,
    "rsi_period": 14,
    "atr_period": 14,
    "ob_lookback": 30,
    "fvg_min_gap_atr": 0.15,
    "liquidity_lookback": 40,
    "equal_level_tolerance_atr": 0.1,
    "risk_per_trade_pct": 1.0,
    "sl_buffer_atr": 0.25,
    "min_rr": 1.5,
    "tp1_rr": 1.5,
    "tp2_rr": 2.5,
    "tp3_rr": 4.0,
    "min_score_to_trade": 60,
    "fib_levels": [0.382, 0.5, 0.618, 0.705, 0.79],
}


Overwriting config.py


In [3]:
%%writefile indicator.py
import pandas as pd
import numpy as np


def ema(series, period):
    return series.ewm(span=period, adjust=False).mean()


def rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi_val = 100 - (100 / (1 + rs))
    return rsi_val.fillna(50)


def atr(df, period=14):
    high, low, close = df["high"], df["low"], df["close"]
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs(),
    ], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / period, adjust=False).mean()


def add_indicators(df, config):
    df = df.copy()
    df["ema_fast"] = ema(df["close"], config["ema_fast"])
    df["ema_slow"] = ema(df["close"], config["ema_slow"])
    df["ema_trend"] = ema(df["close"], config["ema_trend"])
    df["rsi"] = rsi(df["close"], config["rsi_period"])
    df["atr"] = atr(df, config["atr_period"])
    return df


Overwriting indicator.py


In [4]:
%%writefile pattern.py
import pandas as pd


def find_swings(df, lookback=5):
    df = df.copy()
    n = len(df)
    swing_high = [False] * n
    swing_low = [False] * n
    highs = df["high"].values
    lows = df["low"].values

    for i in range(lookback, n - lookback):
        window_high = highs[i - lookback: i + lookback + 1]
        window_low = lows[i - lookback: i + lookback + 1]
        if highs[i] == window_high.max() and (window_high == highs[i]).sum() == 1:
            swing_high[i] = True
        if lows[i] == window_low.min() and (window_low == lows[i]).sum() == 1:
            swing_low[i] = True

    df["swing_high"] = swing_high
    df["swing_low"] = swing_low
    return df


def get_last_swings(df, n_points=4):
    points = []
    for i in range(len(df)):
        if df["swing_high"].iloc[i]:
            points.append({"index": i, "price": df["high"].iloc[i], "type": "high"})
        if df["swing_low"].iloc[i]:
            points.append({"index": i, "price": df["low"].iloc[i], "type": "low"})
    points.sort(key=lambda p: p["index"])
    return points[-n_points:] if len(points) >= n_points else points


Overwriting pattern.py


In [5]:
%%writefile trend.py
import pandas as pd
from pattern import find_swings, get_last_swings


def get_ema_trend(row):
    if row["close"] > row["ema_fast"] > row["ema_slow"] > row["ema_trend"]:
        return "bullish"
    if row["close"] < row["ema_fast"] < row["ema_slow"] < row["ema_trend"]:
        return "bearish"
    return "sideway"


def analyze_structure(df, config):
    df = find_swings(df, lookback=config["swing_lookback"])
    swings = get_last_swings(df, n_points=4)

    result = {
        "trend": "sideway",
        "event": None,
        "event_price": None,
        "last_swings": swings,
        "ema_trend": get_ema_trend(df.iloc[-1]) if len(df) else "sideway",
    }

    if len(swings) < 4:
        return result

    highs = [p for p in swings if p["type"] == "high"]
    lows = [p for p in swings if p["type"] == "low"]

    trend = "sideway"
    if len(highs) >= 2 and len(lows) >= 2:
        higher_high = highs[-1]["price"] > highs[-2]["price"]
        higher_low = lows[-1]["price"] > lows[-2]["price"]
        lower_low = lows[-1]["price"] < lows[-2]["price"]
        lower_high = highs[-1]["price"] < highs[-2]["price"]

        if higher_high and higher_low:
            trend = "bullish"
        elif lower_low and lower_high:
            trend = "bearish"

    result["trend"] = trend

    last_close = df["close"].iloc[-1]
    last_swing_high = highs[-1]["price"] if highs else None
    last_swing_low = lows[-1]["price"] if lows else None

    if trend == "bullish" and last_swing_low is not None and last_close < last_swing_low:
        result["event"] = "CHoCH"
        result["event_price"] = last_swing_low
    elif trend == "bearish" and last_swing_high is not None and last_close > last_swing_high:
        result["event"] = "CHoCH"
        result["event_price"] = last_swing_high
    elif trend == "bullish" and last_swing_high is not None and last_close > last_swing_high:
        result["event"] = "BOS"
        result["event_price"] = last_swing_high
    elif trend == "bearish" and last_swing_low is not None and last_close < last_swing_low:
        result["event"] = "BOS"
        result["event_price"] = last_swing_low

    return result


Overwriting trend.py


In [6]:
%%writefile orderblock.py
import pandas as pd


def find_order_blocks(df, config):
    lookback = config["ob_lookback"]
    start = max(0, len(df) - lookback)
    obs = []

    for i in range(start, len(df) - 1):
        candle = df.iloc[i]
        next_candle = df.iloc[i + 1]
        is_bearish_candle = candle["close"] < candle["open"]
        is_bullish_candle = candle["close"] > candle["open"]
        impulsive_up = next_candle["close"] > candle["high"]
        impulsive_down = next_candle["close"] < candle["low"]

        if is_bearish_candle and impulsive_up:
            obs.append({"type": "bullish", "index": i, "top": candle["high"],
                        "bottom": candle["low"], "mitigated": False})
        elif is_bullish_candle and impulsive_down:
            obs.append({"type": "bearish", "index": i, "top": candle["high"],
                        "bottom": candle["low"], "mitigated": False})

    for ob in obs:
        for j in range(ob["index"] + 1, len(df)):
            price_low = df["low"].iloc[j]
            price_high = df["high"].iloc[j]
            if price_low <= ob["top"] and price_high >= ob["bottom"]:
                ob["mitigated"] = True
                break

    return obs


def get_nearest_unmitigated_ob(obs, direction, current_price):
    candidates = [ob for ob in obs if ob["type"] == direction and not ob["mitigated"]]
    if not candidates:
        return None
    if direction == "bullish":
        below = [ob for ob in candidates if ob["top"] <= current_price]
        if not below:
            return None
        return max(below, key=lambda ob: ob["top"])
    else:
        above = [ob for ob in candidates if ob["bottom"] >= current_price]
        if not above:
            return None
        return min(above, key=lambda ob: ob["bottom"])


Overwriting orderblock.py


In [7]:
%%writefile fvg.py
import pandas as pd


def find_fvgs(df, config):
    fvgs = []
    min_gap = config["fvg_min_gap_atr"]

    for i in range(2, len(df)):
        c1 = df.iloc[i - 2]
        c3 = df.iloc[i]
        atr_val = df["atr"].iloc[i] if "atr" in df.columns else 0

        gap_up = c3["low"] - c1["high"]
        if gap_up > 0 and (atr_val == 0 or gap_up >= min_gap * atr_val):
            fvgs.append({"type": "bullish", "index": i, "top": c3["low"],
                         "bottom": c1["high"], "filled": False})

        gap_down = c1["low"] - c3["high"]
        if gap_down > 0 and (atr_val == 0 or gap_down >= min_gap * atr_val):
            fvgs.append({"type": "bearish", "index": i, "top": c1["low"],
                         "bottom": c3["high"], "filled": False})

    for fvg in fvgs:
        for j in range(fvg["index"] + 1, len(df)):
            price_low = df["low"].iloc[j]
            price_high = df["high"].iloc[j]
            if price_low <= fvg["bottom"] and price_high >= fvg["top"]:
                fvg["filled"] = True
                break

    return fvgs


def get_nearest_unfilled_fvg(fvgs, direction, current_price):
    candidates = [f for f in fvgs if f["type"] == direction and not f["filled"]]
    if not candidates:
        return None
    if direction == "bullish":
        below = [f for f in candidates if f["top"] <= current_price]
        if not below:
            return None
        return max(below, key=lambda f: f["top"])
    else:
        above = [f for f in candidates if f["bottom"] >= current_price]
        if not above:
            return None
        return min(above, key=lambda f: f["bottom"])


Overwriting fvg.py


In [8]:
%%writefile liquidity.py
import pandas as pd
from pattern import find_swings


def find_liquidity_pools(df, config):
    lookback = config["liquidity_lookback"]
    tolerance_mult = config["equal_level_tolerance_atr"]

    sub_df = df.iloc[-lookback:].copy() if len(df) > lookback else df.copy()
    sub_df = find_swings(sub_df, lookback=config["swing_lookback"])

    atr_val = df["atr"].iloc[-1] if "atr" in df.columns and not pd.isna(df["atr"].iloc[-1]) else df["close"].iloc[-1] * 0.001
    tolerance = tolerance_mult * atr_val

    highs = sub_df.loc[sub_df["swing_high"], "high"].tolist()
    lows = sub_df.loc[sub_df["swing_low"], "low"].tolist()

    def cluster(levels):
        clusters = []
        used = [False] * len(levels)
        for i, lvl in enumerate(levels):
            if used[i]:
                continue
            group = [lvl]
            used[i] = True
            for j in range(i + 1, len(levels)):
                if not used[j] and abs(levels[j] - lvl) <= tolerance:
                    group.append(levels[j])
                    used[j] = True
            if len(group) >= 2:
                clusters.append(sum(group) / len(group))
        return clusters

    eqh = cluster(highs)
    eql = cluster(lows)
    return {"equal_highs": eqh, "equal_lows": eql}


Overwriting liquidity.py


In [9]:
%%writefile fibo.py
def calc_fib_levels(swing_low_price, swing_high_price, direction, config):
    diff = swing_high_price - swing_low_price
    levels = {}
    for lvl in config["fib_levels"]:
        if direction == "bullish":
            levels[lvl] = swing_high_price - diff * lvl
        else:
            levels[lvl] = swing_low_price + diff * lvl
    return levels


def is_in_ote_zone(price, fib_levels, low_bound=0.618, high_bound=0.79):
    prices = [p for lvl, p in fib_levels.items() if low_bound <= lvl <= high_bound]
    if not prices:
        return False
    return min(prices) <= price <= max(prices)


Overwriting fibo.py


In [10]:
%%writefile entry.py
from orderblock import find_order_blocks, get_nearest_unmitigated_ob
from fvg import find_fvgs, get_nearest_unfilled_fvg
from liquidity import find_liquidity_pools
from fibo import calc_fib_levels, is_in_ote_zone


def evaluate_entry(df, structure, config):
    current_price = df["close"].iloc[-1]
    trend = structure["trend"]
    event = structure["event"]

    result = {
        "valid": False, "direction": None, "entry_price": None,
        "reasons": [], "ob": None, "fvg": None, "liquidity": None, "fib_levels": None,
    }

    if trend == "sideway" or event is None:
        result["reasons"].append("ตลาดยังไม่มีโครงสร้างชัดเจน (sideway หรือไม่มี BOS/CHoCH) — ข้าม")
        return result

    direction = "bullish" if trend == "bullish" else "bearish"
    result["direction"] = direction
    result["reasons"].append(f"โครงสร้างตลาดเป็น {trend} และเพิ่งเกิด {event} ยืนยันทิศทาง")

    obs = find_order_blocks(df, config)
    ob = get_nearest_unmitigated_ob(obs, direction, current_price)
    if ob:
        result["ob"] = ob
        result["reasons"].append(f"พบ {direction} Order Block ที่ยังไม่ถูกแตะ บริเวณ {ob['bottom']:.4f}-{ob['top']:.4f}")

    fvgs = find_fvgs(df, config)
    fvg = get_nearest_unfilled_fvg(fvgs, direction, current_price)
    if fvg:
        result["fvg"] = fvg
        result["reasons"].append(f"พบ {direction} FVG ที่ยังไม่ถูกเติมเต็ม บริเวณ {fvg['bottom']:.4f}-{fvg['top']:.4f}")

    liquidity = find_liquidity_pools(df, config)
    result["liquidity"] = liquidity
    if direction == "bullish" and liquidity["equal_lows"]:
        result["reasons"].append("มี Equal Lows (liquidity) ด้านล่างที่อาจถูกกวาดมาแล้วก่อนกลับตัวขึ้น")
    if direction == "bearish" and liquidity["equal_highs"]:
        result["reasons"].append("มี Equal Highs (liquidity) ด้านบนที่อาจถูกกวาดมาแล้วก่อนกลับตัวลง")

    swings = structure["last_swings"]
    highs = [p for p in swings if p["type"] == "high"]
    lows = [p for p in swings if p["type"] == "low"]
    if highs and lows:
        swing_high_price = highs[-1]["price"]
        swing_low_price = lows[-1]["price"]
        fib_levels = calc_fib_levels(swing_low_price, swing_high_price, direction, config)
        result["fib_levels"] = fib_levels
        if is_in_ote_zone(current_price, fib_levels):
            result["reasons"].append("ราคาปัจจุบันอยู่ในโซน OTE (Fib 0.618-0.79) ซึ่งเป็นโซน entry ที่ดี")

    if ob or fvg:
        result["valid"] = True
        zone_edges = []
        if ob:
            zone_edges.append(ob["top"] if direction == "bullish" else ob["bottom"])
        if fvg:
            zone_edges.append(fvg["top"] if direction == "bullish" else fvg["bottom"])
        result["entry_price"] = sum(zone_edges) / len(zone_edges)
    else:
        result["reasons"].append("ไม่พบ Order Block หรือ FVG รองรับ — ยังไม่แนะนำเข้า")

    return result


Overwriting entry.py


In [11]:
%%writefile risk.py
def calc_stop_loss(entry_signal, current_atr, config):
    direction = entry_signal["direction"]
    buffer = config["sl_buffer_atr"] * current_atr

    ob = entry_signal.get("ob")
    fvg = entry_signal.get("fvg")

    if ob:
        base = ob["bottom"] if direction == "bullish" else ob["top"]
    elif fvg:
        base = fvg["bottom"] if direction == "bullish" else fvg["top"]
    else:
        base = entry_signal["entry_price"]

    if direction == "bullish":
        return base - buffer
    else:
        return base + buffer


def calc_position_size(account_balance, entry_price, stop_loss, config):
    risk_amount = account_balance * (config["risk_per_trade_pct"] / 100)
    sl_distance = abs(entry_price - stop_loss)
    if sl_distance == 0:
        return {"risk_amount": risk_amount, "sl_distance": 0, "position_size": 0}
    position_size = risk_amount / sl_distance
    return {
        "risk_amount": round(risk_amount, 2),
        "sl_distance": round(sl_distance, 6),
        "position_size": round(position_size, 6),
    }


Overwriting risk.py


In [12]:
%%writefile tp.py
def calc_take_profits(entry_price, stop_loss, direction, config):
    risk_distance = abs(entry_price - stop_loss)
    tp_rr = {"TP1": config["tp1_rr"], "TP2": config["tp2_rr"], "TP3": config["tp3_rr"]}
    tps = {}
    for name, rr in tp_rr.items():
        if direction == "bullish":
            tps[name] = entry_price + risk_distance * rr
        else:
            tps[name] = entry_price - risk_distance * rr
    return tps


def calc_risk_reward(entry_price, stop_loss, tp_price):
    risk = abs(entry_price - stop_loss)
    reward = abs(tp_price - entry_price)
    if risk == 0:
        return 0.0
    return round(reward / risk, 2)


Overwriting tp.py


In [13]:
%%writefile score.py
WEIGHTS = {
    "structure_event": 20, "order_block": 20, "fvg": 15,
    "liquidity_sweep": 15, "ote_zone": 15, "rsi_confirm": 10, "rr_quality": 5,
}


def calc_confidence_score(entry_signal, structure, df, config, rr_tp1):
    score = 0
    breakdown = {}

    if structure.get("event") in ("BOS", "CHoCH"):
        score += WEIGHTS["structure_event"]
        breakdown["structure_event"] = WEIGHTS["structure_event"]

    if entry_signal.get("ob"):
        score += WEIGHTS["order_block"]
        breakdown["order_block"] = WEIGHTS["order_block"]

    if entry_signal.get("fvg"):
        score += WEIGHTS["fvg"]
        breakdown["fvg"] = WEIGHTS["fvg"]

    liquidity = entry_signal.get("liquidity") or {}
    direction = entry_signal.get("direction")
    if direction == "bullish" and liquidity.get("equal_lows"):
        score += WEIGHTS["liquidity_sweep"]
        breakdown["liquidity_sweep"] = WEIGHTS["liquidity_sweep"]
    elif direction == "bearish" and liquidity.get("equal_highs"):
        score += WEIGHTS["liquidity_sweep"]
        breakdown["liquidity_sweep"] = WEIGHTS["liquidity_sweep"]

    fib_levels = entry_signal.get("fib_levels")
    if fib_levels:
        from fibo import is_in_ote_zone
        current_price = df["close"].iloc[-1]
        if is_in_ote_zone(current_price, fib_levels):
            score += WEIGHTS["ote_zone"]
            breakdown["ote_zone"] = WEIGHTS["ote_zone"]

    rsi_val = df["rsi"].iloc[-1] if "rsi" in df.columns else 50
    if direction == "bullish" and rsi_val < 65:
        score += WEIGHTS["rsi_confirm"]
        breakdown["rsi_confirm"] = WEIGHTS["rsi_confirm"]
    elif direction == "bearish" and rsi_val > 35:
        score += WEIGHTS["rsi_confirm"]
        breakdown["rsi_confirm"] = WEIGHTS["rsi_confirm"]

    if rr_tp1 >= config["min_rr"]:
        score += WEIGHTS["rr_quality"]
        breakdown["rr_quality"] = WEIGHTS["rr_quality"]

    return {"score": score, "breakdown": breakdown}


Overwriting score.py


In [14]:
%%writefile report.py
def print_report(symbol, timeframe, structure, entry_signal, stop_loss,
                  take_profits, position, rr, confidence, config):

    print("=" * 60)
    print(f"  TRADE SETUP: {symbol} | TF: {timeframe}")
    print("=" * 60)

    if not entry_signal["valid"]:
        print("สถานะ: ยังไม่มีจังหวะเข้าไม้ที่น่าสนใจ")
        print("-" * 60)
        for r in entry_signal["reasons"]:
            print(f" • {r}")
        print("=" * 60)
        return

    direction_th = "LONG (ซื้อ)" if entry_signal["direction"] == "bullish" else "SHORT (ขาย)"
    score = confidence["score"]
    passed = score >= config["min_score_to_trade"]

    print(f"ทิศทาง: {direction_th}")
    print(f"เทรนด์/Event: {structure['trend']} | {structure['event']}")
    print(f"คะแนนความมั่นใจ: {score}/100  -> {'ผ่านเกณฑ์' if passed else 'ต่ำกว่าเกณฑ์ แนะนำรอดูก่อน'}")
    print("-" * 60)

    print(f"Entry ที่แนะนำ : {entry_signal['entry_price']:.4f}")
    print(f"Stop Loss      : {stop_loss:.4f}")
    for name, price in take_profits.items():
        print(f"{name:<15}: {price:.4f}  (RR {rr[name]})")
    print("-" * 60)

    print(f"ขนาดโพซิชั่นแนะนำ : {position['position_size']}")
    print(f"เงินที่เสี่ยง       : {position['risk_amount']}")
    print("-" * 60)

    print("เหตุผลประกอบการวิเคราะห์:")
    for r in entry_signal["reasons"]:
        print(f" • {r}")

    print("รายละเอียดคะแนน:")
    for factor, pts in confidence["breakdown"].items():
        print(f" • {factor}: +{pts}")

    print("=" * 60)


Overwriting report.py


In [15]:
%%writefile main.py
import pandas as pd
import numpy as np

from config import CONFIG
from indicator import add_indicators
from trend import analyze_structure
from entry import evaluate_entry
from risk import calc_stop_loss, calc_position_size
from tp import calc_take_profits, calc_risk_reward
from score import calc_confidence_score
from report import print_report


def load_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.lower() for c in df.columns]
    required = {"open", "high", "low", "close", "volume"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV ขาดคอลัมน์: {missing}")
    return df.reset_index(drop=True)


def generate_synthetic_data(n=400, seed=42):
    rng = np.random.default_rng(seed)
    price = 100.0
    rows = []
    trend_bias = 0.0
    for i in range(n):
        if i % 60 == 0:
            trend_bias = rng.choice([-0.15, 0.15, 0.0])
        change = rng.normal(loc=trend_bias, scale=1.0)
        open_p = price
        close_p = price + change
        high_p = max(open_p, close_p) + abs(rng.normal(0, 0.5))
        low_p = min(open_p, close_p) - abs(rng.normal(0, 0.5))
        volume = rng.integers(100, 1000)
        rows.append([open_p, high_p, low_p, close_p, volume])
        price = close_p
    return pd.DataFrame(rows, columns=["open", "high", "low", "close", "volume"])


def run_pipeline(df, symbol="SYMBOL", timeframe="15m", account_balance=1000.0, config=CONFIG):
    df = add_indicators(df, config)
    structure = analyze_structure(df, config)
    entry_signal = evaluate_entry(df, structure, config)

    if not entry_signal["valid"]:
        print_report(symbol, timeframe, structure, entry_signal,
                      stop_loss=None, take_profits={}, position={},
                      rr={}, confidence={"score": 0, "breakdown": {}}, config=config)
        return

    current_atr = df["atr"].iloc[-1]
    stop_loss = calc_stop_loss(entry_signal, current_atr, config)
    take_profits = calc_take_profits(entry_signal["entry_price"], stop_loss,
                                      entry_signal["direction"], config)
    rr = {name: calc_risk_reward(entry_signal["entry_price"], stop_loss, price)
          for name, price in take_profits.items()}
    position = calc_position_size(account_balance, entry_signal["entry_price"], stop_loss, config)
    confidence = calc_confidence_score(entry_signal, structure, df, config, rr["TP1"])

    if rr["TP1"] < config["min_rr"]:
        entry_signal["reasons"].append(
            f"RR ของ TP1 ({rr['TP1']}) ต่ำกว่าเกณฑ์ขั้นต่ำ ({config['min_rr']}) — ไม่แนะนำเข้า"
        )
        entry_signal["valid"] = False

    print_report(symbol, timeframe, structure, entry_signal, stop_loss,
                 take_profits, position, rr, confidence, config)


if __name__ == "__main__":
    sample_df = generate_synthetic_data(n=400)
    run_pipeline(sample_df, symbol="BTCUSDT (SAMPLE)", timeframe="15m", account_balance=1000)


Overwriting main.py


In [18]:
%%writefile fetch_data.py
import requests
import pandas as pd


def fetch_binance(symbol="BTCUSDT", interval="15m", limit=500):
    url = "https://api.binance.com/api/v3/klines"
    params = dict(symbol=symbol.upper(), interval=interval, limit=limit)
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    raw = resp.json()

    df = pd.DataFrame(raw, columns=[
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "qav", "trades", "tbbav", "tbqav", "ignore",
    ])
    df = df[["open", "high", "low", "close", "volume"]].astype(float)
    return df.reset_index(drop=True)


def fetch_yahoo(symbol="GC=F", interval="15m", range_="5d"):
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
    params = dict(interval=interval, range=range_)
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(url, params=params, headers=headers, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    result = data["chart"]["result"][0]
    quote = result["indicators"]["quote"][0]

    df = pd.DataFrame({
        "open": quote["open"],
        "high": quote["high"],
        "low": quote["low"],
        "close": quote["close"],
        "volume": quote["volume"],
    })
    df = df.dropna().reset_index(drop=True)
    return df


Writing fetch_data.py


In [19]:
!ls


Trading bot.ipynb
Untitled.ipynb
config.py
entry.py
fetch_data.py
fibo.py
fvg.py
indicator.py
liquidity.py
main.py
orderblock.py
pattern.py
report.py
risk.py
score.py
tp.py
trading bot.ipynb
trend.py
welcome


In [1]:
%run main.py


  TRADE SETUP: BTCUSDT (SAMPLE) | TF: 15m
ทิศทาง: SHORT (ขาย)
เทรนด์/Event: bearish | BOS
คะแนนความมั่นใจ: 40/100  -> ต่ำกว่าเกณฑ์ แนะนำรอดูก่อน
------------------------------------------------------------
Entry ที่แนะนำ : 76.1802
Stop Loss      : 77.6977
TP1            : 73.9039  (RR 1.5)
TP2            : 72.3864  (RR 2.5)
TP3            : 70.1102  (RR 4.0)
------------------------------------------------------------
ขนาดโพซิชั่นแนะนำ : 6.589804
เงินที่เสี่ยง       : 10.0
------------------------------------------------------------
เหตุผลประกอบการวิเคราะห์:
 • โครงสร้างตลาดเป็น bearish และเพิ่งเกิด BOS ยืนยันทิศทาง
 • พบ bearish FVG ที่ยังไม่ถูกเติมเต็ม บริเวณ 76.1802-77.2100
รายละเอียดคะแนน:
 • structure_event: +20
 • fvg: +15
 • rr_quality: +5


In [ ]:
import main
df = main.load_csv("ชื่อไฟล์ของคุณ.csv")
main.run_pipeline(df, symbol="XAUUSD", timeframe="15m", account_balance=1000)
